In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

from starter_code.data_utils import get_images, get_labels

## Load All Data

In [ ]:
disaster = "hurricane-matthew"
data = {}
split = "train"

with open("config.json") as config_file:
    config = json.load(config_file)
    data_dir = config["data_dir"]

print(f"Loading {split} images and labels for {disaster} dataset...")
images = get_images(data_dir, disaster, split=split)
labels = get_labels(data_dir, disaster, split=split)
data[disaster] = {"images": images, "labels": labels}

labels = np.array(labels)

print("Number of images:", len(images))
print("Label distribution:", {
    k: int(v) for k, v in pd.Series(labels).value_counts().sort_index().to_dict().items()
})


# Multi-Classification: Hurricane vs. Damage Level

## Process Data

In [ ]:
images = np.array(images)

images_normalized = images / 255.0
rgb_means = np.mean(images_normalized, axis=(1, 2))

X = rgb_means
Y = labels

print("Feature shape (X):", X.shape)
print("Labels shape (Y):", Y.shape)

In [ ]:
X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y,test_size=0.2, stratify=Y, random_state=42)

print("Train size:", X_train.shape[0])
print("Valid size:", X_valid.shape[0])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
def evaluate_on_validation(name, model, X_valid, Y_valid):
    """
    Print accuracy, weighted F1, classification report, and show confusion matrix for a trained model evaluated on the validation set.
    """
    valid_preds = model.predict(X_valid)

    acc = accuracy_score(Y_valid, valid_preds)
    f1 = f1_score(Y_valid, valid_preds, average="weighted")

    print(f"{name} - Validation Performance")
    print("Accuracy:", acc)
    print("Weighted F1:", f1)
    print("Classification Report:")
    print(classification_report(Y_valid, valid_preds))

    cm = confusion_matrix(Y_valid, valid_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name} - Confusion Matrix (Validation)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    return acc, f1

## Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, multi_class="multinomial", class_weight="balanced", solver="lbfgs", random_state=42)

log_reg_param_grid = {"C": [0.01, 0.1, 1, 10]}

log_reg_grid = GridSearchCV(estimator=log_reg, param_grid=log_reg_param_grid, cv=cv, scoring="f1_weighted", n_jobs=-1, verbose=1)

log_reg_grid.fit(X_train, Y_train)

print("Best params (LogReg):", log_reg_grid.best_params_)
print("Best CV Weighted F1:", log_reg_grid.best_score_)

best_log_reg = log_reg_grid.best_estimator_
logreg_val_acc, logreg_val_f1 = evaluate_on_validation("Logistic Regression", best_log_reg, X_valid, Y_valid)

## Support Vector Machine

In [ ]:
from sklearn.svm import SVC

svm = SVC(class_weight="balanced")

svm_param_grid = {"C": [0.1, 1, 10], "gamma": ["scale", 0.01, 0.001], "kernel": ["rbf"]}

svm_grid = GridSearchCV(
    estimator=svm,
    param_grid=svm_param_grid,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train, Y_train)

print("Best params:", svm_grid.best_params_)
print("Best CV macro F1:", svm_grid.best_score_)

best_svm = svm_grid.best_estimator_
svm_val_acc, svm_val_f1 = evaluate_on_validation("SVM (RBF)", best_svm, X_valid, Y_valid)

## Boosted Trees

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

gb_param_grid = {"n_estimators": [50, 100, 200],"learning_rate": [0.05, 0.1, 0.2],"max_depth": [2, 3, 5]}

gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=gb_param_grid,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

gb_grid.fit(X_train, Y_train)

print("Best params (GB):", gb_grid.best_params_)
print("Best CV Weighted F1 (GB):", gb_grid.best_score_)

best_gb = gb_grid.best_estimator_
gb_val_acc, gb_val_f1 = evaluate_on_validation(
    "Gradient Boosting",
    best_gb,
    X_valid,
    Y_valid
)

In [ ]:
results = {
    "Logistic Regression": {
        "cv_f1_weighted": log_reg_grid.best_score_,
        "val_accuracy": logreg_val_acc,
        "val_f1_weighted": logreg_val_f1,
    },
    "SVM (RBF)": {
        "cv_f1_weighted": svm_grid.best_score_,
        "val_accuracy": svm_val_acc,
        "val_f1_weighted": svm_val_f1,
    },
    "Gradient Boosting": {
        "cv_f1_weighted": gb_grid.best_score_,
        "val_accuracy": gb_val_acc,
        "val_f1_weighted": gb_val_f1,
    },
}

print("Model Comparison : Hurricane vs Damage Level")
for model_name, scores in results.items():
    print(f"\n{model_name}:")
    for k, v in scores.items():
        print(f"  {k}: {v}")